## 1. Preprocessing Pipeline.
Define a unified preprocessing pipeline for all experiments at 256x256 RGB, compatible with ImageNet-pretrained models.
#### Rationale:
- Standarise inputs across CheXpert (grayscale, variable size) and TBX11K (RGB, 512x512).
- Enable fair comparison among custom CNN and transfer-learning baselines.
- Use ImageNet normalisation to align with pretrained weights.
- Allow an easy switch to 320x320 for the pre-registered sensitivity analysis.

In [2]:
# Define a reproducible preprocessing pipeline for training and validation/testing.
# Baseline: 256x256 RGB for all datasets; optional 320x320 sensitivity via IMG_SIZE variable.

from pathlib import Path
from torchvision import transforms as T

# Project root inferred from this notebook living in notebooks/
PROJECT_ROOT = Path("..").resolve()

# Toggle here for running the sensitivity analysis.
IMG_SIZE = 256 # Set to 320 for the sensitivity analysis.

# ImageNet normalisation (required for the MobileNetV2, EfficientNet-B0, ResNet-50 pretrained weights).
# Rescale pixel values between CheXpert and TBX11K datasets, so they are centred and comparable.
# During that training, inputs were normalised with specific mean and standard deviation values.
# Apply ImageNet normalisation to meet what pretrained filters expect.
IMAGENET_MEAN = [0.485, 0.456, 0.406]  # MobileNetV2, EfficientNet-B0, and ResNet-50 were pretrained on
IMAGENET_STD  = [0.229, 0.224, 0.225] # ImageNet with these exact statistics.

# Training-time augmentations (mild, medically, sensible: avoid aggressive geometric distortions).
train_transforms_chexpert = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),     # Standarise spatial size (IMG_SIZE X IMG_SIZE).
    T.Grayscale(num_output_channels=3), # CheXpert is L (1ch): replicate to RGB (3ch).
    T.RandomHorizontalFlip(p=0.5),      # Lungs are roughly symmetric; safe augmentation.
    T.RandomRotation(degrees=5),        # Small rotations to improve rebustness.
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_transforms_tbx11k = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),   # TBX11K is 512x512 RGB.
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=5),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Validation/test transforms (no stochastic augmentation). Only standarise size, convert to grey scale, to tenson, and normalise to match 
# ImageNet stats.
eval_transforms_chexpert = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.Grayscale(num_output_channels=3),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

eval_transforms_tbx11k = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print(f'Preprocessing ready at {IMG_SIZE}x{IMG_SIZE} RGB.')
print(f'- CheXpert: grayscale -> RGB + normalisation.')
print(f'- TBX11K  : RGB + normalisation.')

# Clarify the modelling classes and their data sources (for logging/record-keeping).
print("\nClasses will be used for training: \n- 'Pneumonia' (CheXpert frontal-only)\n- 'Normal' (TBX11K) \n- 'TB' (TBX11K)")

Preprocessing ready at 256x256 RGB.
- CheXpert: grayscale -> RGB + normalisation.
- TBX11K  : RGB + normalisation.

Classes will be used for training: 
- 'Pneumonia' (CheXpert frontal-only)
- 'Normal' (TBX11K) 
- 'TB' (TBX11K)


# 2. Dataset scaffolding.
Build a unified three-class Dataset using RELATIVE paths, mirroring 01_data_eda.ipynb behavior.
Store relative paths and a 'Source' tag in a DataFrame; compose absolute paths lazily in `__getitem__`.
#### Target class order (fixed across the whole project):
- 0 : 'Pneumonia' (CheXpert frontal-only)
- 1 : 'TB'        (TBX11K)
- 2 : 'Normal'    (TBX11K)
#### Rationale:
- Keep paths relative to dataset roots for portability and reproducibility.
- Enforce RGB at load time to align with ImageNet-pretrained models.
- Avoid RAM preloading; open images on demand for efficiency.

In [ ]:
# Build a unified three-class Dataset using RELATIVE path (mirror 01_data_eda.ipynb behavior).
# Store relative paths and a source tag in the DataFrame; compose absolute paths in __getitem__.
# Target classes: Pneumonia (CheXpert frontal-only), TB (TBX11K), Normal (TBX11K).

from pathlib import Path
from typing import List, Dict
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset

# Stable label mapping used everywhere in training and evaluation.
# Fix the label order explicitely for all downstram components.
CLASS_TO_INDEX: Dict[str, int] = {"Pneumonia": 0, "TB": 1, "Normal": 2}
# Create the inverse mapping (index -> class) for readability, debugging, and plotting.
INDEX_TO_CLASS = {v: k for k, v in CLASS_TO_INDEX.items()}

def make_chexpert_pneumonia_frontal_df_rel(chexpert_root: Path) -> pd.DataFrame:
    """
    Build a DataFrame for CheXpert Pneumonia (frontal-only) with RELATIVE paths.
    Returns columns: ['rel_path', 'Class", 'Source'] with Class == 'Pneumonia' and Source == 'chexpert'.
    """
    # Load the CheXpert metadata (train.csv) which includes labels for 14 pathologies.
    meta = pd.read_csv(chexpert_root / "train.csv")

    # Keep only the file path and the "Pneumonia" column, discarding all other labels.
    df = meta[["Path", "Pneumonia"]].copy()

    # Remove the original dataset prefix ("CheXpert-v1.0-small/") to align with the extracted folder structure
    # in the repository. This makes paths portable.
    df["Path"] = df["Path"].str.replace("CheXpert-v1.0-small/", "", regex=False)